In [17]:
import json
import gzip
import pandas as pd

# Configuration
MATRIX_PATH = "/content/Galaxy4430-[computeMatrix on dataset 4337-4344 and 4428 and collection 4345_ Matrix].deeptools_compute_matrix_archive"
OUTPUT_CSV = "deeptools_matrix_full.csv"

def _is_gz(path):
    with open(path, "rb") as f:
        return f.read(2) == b"\x1f\x8b"

def _open(path):
    if path.endswith(".gz") or _is_gz(path):
        return gzip.open(path, "rt")
    return open(path, "r")


In [18]:
# 1. Parse the JSON header to get sample labels for column naming
with _open(MATRIX_PATH) as f:
    header = json.loads(f.readline().lstrip("@").strip())

sample_labels = header["sample_labels"]
sample_boundaries = header["sample_boundaries"]
bin_size = header["bin size"][0]


In [19]:
# 2. Load the data block
with _open(MATRIX_PATH) as f:
    # Skip the first header line
    df_matrix = pd.read_csv(f, sep="\t", header=None, skiprows=1)

In [20]:
# 3. Create column names
# Standard deepTools matrix columns: 0: chrom, 1: start, 2: end, 3: name, 4: score, 5: strand
base_cols = ["chrom", "start", "end", "name", "score", "strand"]

# Generate bin-specific labels for each sample
signal_cols = []
for i, label in enumerate(sample_labels):
    n_bins = sample_boundaries[i+1] - sample_boundaries[i]
    for b in range(n_bins):
        signal_cols.append(f"{label}_bin_{b}")

df_matrix.columns = base_cols + signal_cols

In [21]:
# 4. Save to CSV
df_matrix.to_csv(OUTPUT_CSV, index=False)
print(f"Success! Full matrix converted and saved to {OUTPUT_CSV}")


Success! Full matrix converted and saved to deeptools_matrix_full.csv


In [22]:
df_matrix.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1556 entries, 0 to 1555
Columns: 646 entries, chrom to bamCoverage on S6-HER_bin_79
dtypes: float64(641), int64(2), object(3)
memory usage: 7.7+ MB
